In [1]:
# Cargar de librerias 
import pandas as pd
import numpy as np
import plotly.express as px
import scipy
from pathlib import Path


In [2]:
# Carga de datos 

df_ia = pd.read_csv("../data/raw/occ_level.csv")
# df_cw_soc = pd.read_csv("../data/raw/crosswalks/soc_data.csv", dtype=str)
# df_cw_cps = pd.read_csv("../data/raw/crosswalks/cps_2020_data.csv", dtype=str)
df_bls = pd.read_csv("../data/raw/bls_oews_national.csv", dtype=str)
### df_cw = pd.read_csv("../data/raw/census_crosswalk.csv", dtype=str) -> Se carga de forma especial mas adelante 
df_cps = pd.read_csv("../data/raw/cps_00001.csv")


In [3]:
#Solo traer los los detallados

if "O_GROUP" in df_bls.columns:
    df_bls = df_bls[df_bls["O_GROUP"] == "detailed"].copy()

In [4]:
# Limpieza de variables a usar 

df_bls["TOT_EMP"] = pd.to_numeric(df_bls["TOT_EMP"].str.replace(",", ""), errors="coerce")
df_bls["A_MEDIAN"] = pd.to_numeric(df_bls["A_MEDIAN"].str.replace(",", ""), errors="coerce")
df_bls["SOC_6dig"] = df_bls["OCC_CODE"].str.strip()

OCC_TITLE: Nombre oficial de la ocupación según el Bureau of Labor Statistics (BLS).

TOT_EMP: Empleo total estimado a nivel nacional en Estados Unidos para esa ocupación (número total de trabajadores ocupados en ese cargo en todo el país).

A_MEDIAN: Salario anual mediano en dólares estadounidenses para esa ocupación a nivel nacional. Representa el punto medio exacto donde el 50 % de los trabajadores gana más y el 50 % gana menos (es más robusto que el promedio ante salarios extremos).   

In [5]:
df_bls_limpio = df_bls[["SOC_6dig", "OCC_TITLE", "TOT_EMP", "A_MEDIAN"]].dropna(subset=["SOC_6dig"])

In [6]:
df = pd.read_csv(
    '../data/raw/census_crosswalk.csv',
    usecols=[0, 1, 2, 3],
    skiprows=8,
    header=None,
    dtype=str,
)

df.columns = ['Major_Group', 'Census_Title', 'Census_Code', 'SOC_Code']

In [7]:
df = df.dropna(how='all').copy()

In [8]:
df['Major_Group'] = df['Major_Group'].str.rstrip(':').ffill()
df_clean = df[df['Census_Code'].str.match(r'^\d{4}$', na=False)].copy()
for col in df_clean.columns:
  df_clean[col] = df_clean[col].str.strip()

df_cw = df_clean.reset_index(drop=True)

df_cw

,Major_Group,Census_Title,Census_Code,SOC_Code
0,"Management, Business, Science, and Arts Occup...",Chief executives,0010,11-1011
1,"Management, Business, Science, and Arts Occup...",General and operations managers,0020,11-1021
2,"Management, Business, Science, and Arts Occup...",Legislators,0030,11-1031
3,"Management, Business, Science, and Arts Occup...",Advertising and promotions managers,0040,11-2011
4,"Management, Business, Science, and Arts Occup...",Marketing managers,0051,11-2021
...,...,...,...,...
565,Military Specific Occupations,Military officer special and tactical operatio...,9800,55-1010
566,Military Specific Occupations,First-line enlisted military supervisors,9810,55-2010
567,Military Specific Occupations,Military enlisted tactical operations and air/...,9825,55-3010
568,Military Specific Occupations,"Military, rank not Specified",9830,none


In [9]:
df_cps['OCC_clean'] = (
    df_cps['OCC'].fillna(0).astype(int).astype(str).str.zfill(4)
)
df_cw['Census_Code'] = df_cw['Census_Code'].astype(str).str.zfill(4)

In [10]:
df_merged = pd.merge(
    df_cps,
    df_cw,
    left_on='OCC_clean',
    right_on='Census_Code',
    how='left',
)

df_merged

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,EMPSTAT,OCC,OCC2010,DURUNEMP,EDUC,OCC_clean,Major_Group,Census_Title,Census_Code,SOC_Code
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,32,0,9999,999,81,0000,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,10,4150,4150,999,73,4150,Service Occupations,"Hosts and hostesses, restaurant, lounge, and c...",4150,35-9031
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,21,4110,4110,39,111,4110,Service Occupations,Waiters and waitresses,4110,35-3031
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,36,0,9999,999,123,0000,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,10,3602,4610,999,73,3602,Service Occupations,Personal care aides,3602,31-1122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,10,850,850,999,111,0850,"Management, Business, Science, and Arts Occup...",Personal financial advisors,0850,13-2052
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,10,3930,3930,999,123,3930,Service Occupations,Security guards and gambling surveillance offi...,3930,33-9030
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,10,3603,3600,999,92,3603,Service Occupations,Nursing assistants,3603,31-1131
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,10,2014,2010,999,123,2014,"Management, Business, Science, and Arts Occup...","Social workers, all other",2014,21-1029


In [11]:
df_merged = df_merged.drop(columns=['OCC_clean', 'Census_Code'])

In [12]:
df_merged['SOC_Code'] = df_merged['SOC_Code'].astype(str).str.strip()
df_bls_limpio['SOC_6dig'] = df_bls_limpio['SOC_6dig'].astype(str).str.strip()

In [13]:
df_final = pd.merge(
    df_merged,
    df_bls_limpio,
    left_on='SOC_Code',
    right_on='SOC_6dig',
    how='left',
)
df_final

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,OCC2010,DURUNEMP,EDUC,Major_Group,Census_Title,SOC_Code,SOC_6dig,OCC_TITLE,TOT_EMP,A_MEDIAN
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,9999,999,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,4150,999,73,Service Occupations,"Hosts and hostesses, restaurant, lounge, and c...",35-9031,35-9031,"Hosts and Hostesses, Restaurant, Lounge, and C...",432690.0,31200.0
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,4110,39,111,Service Occupations,Waiters and waitresses,35-3031,35-3031,Waiters and Waitresses,2270910.0,35230.0
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,9999,999,123,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,4610,999,73,Service Occupations,Personal care aides,31-1122,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,850,999,111,"Management, Business, Science, and Arts Occup...",Personal financial advisors,13-2052,13-2052,Personal Financial Advisors,266800.0,105070.0
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,3930,999,123,Service Occupations,Security guards and gambling surveillance offi...,33-9030,NaN,NaN,NaN,NaN
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,3600,999,92,Service Occupations,Nursing assistants,31-1131,31-1131,Nursing Assistants,1448910.0,42260.0
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,2010,999,123,"Management, Business, Science, and Arts Occup...","Social workers, all other",21-1029,21-1029,"Social Workers, All Other",62930.0,71900.0


In [14]:
df_bls_limpio['SOC_prefijo'] = df_bls_limpio['SOC_6dig'].str[:6]
bls_grupo = df_bls_limpio.groupby('SOC_prefijo', as_index=False).agg({
    'TOT_EMP': 'sum',
    'A_MEDIAN': 'mean',
})

In [15]:
df_final['SOC_prefijo'] = df_final['SOC_Code'].str[:6]
df_final = pd.merge(
    df_final, bls_grupo, on='SOC_prefijo', how='left', suffixes=('', '_grupo')
)
df_final

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,Major_Group,Census_Title,SOC_Code,SOC_6dig,OCC_TITLE,TOT_EMP,A_MEDIAN,SOC_prefijo,TOT_EMP_grupo,A_MEDIAN_grupo
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,Service Occupations,"Hosts and hostesses, restaurant, lounge, and c...",35-9031,35-9031,"Hosts and Hostesses, Restaurant, Lounge, and C...",432690.0,31200.0,35-903,432690.0,31200.0
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,Service Occupations,Waiters and waitresses,35-3031,35-3031,Waiters and Waitresses,2270910.0,35230.0,35-303,2270910.0,35230.0
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,Service Occupations,Personal care aides,31-1122,NaN,NaN,NaN,NaN,31-112,4305810.0,35800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,"Management, Business, Science, and Arts Occup...",Personal financial advisors,13-2052,13-2052,Personal Financial Advisors,266800.0,105070.0,13-205,798050.0,101627.5
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,Service Occupations,Security guards and gambling surveillance offi...,33-9030,NaN,NaN,NaN,NaN,33-903,1292990.0,40695.0
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,Service Occupations,Nursing assistants,31-1131,31-1131,Nursing Assistants,1448910.0,42260.0,31-113,1536870.0,41820.0
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,"Management, Business, Science, and Arts Occup...","Social workers, all other",21-1029,21-1029,"Social Workers, All Other",62930.0,71900.0,21-102,775920.0,64902.5


In [16]:
df_final['TOT_EMP'] = df_final['TOT_EMP'].fillna(df_final['TOT_EMP_grupo'])
df_final['A_MEDIAN'] = df_final['A_MEDIAN'].fillna(df_final['A_MEDIAN_grupo'])

In [17]:
df_final = df_final.drop(
    columns=['SOC_6dig', 'SOC_prefijo', 'TOT_EMP_grupo', 'A_MEDIAN_grupo']
)

In [18]:
df_final

,YEAR,SERIAL,MONTH,HWTFINL,CPSID,ASECFLAG,ASECWTH,PERNUM,WTFINL,CPSIDP,...,OCC,OCC2010,DURUNEMP,EDUC,Major_Group,Census_Title,SOC_Code,OCC_TITLE,TOT_EMP,A_MEDIAN
0,2021,1,1,1805.3669,20191100000100,NaN,NaN,1,1805.3669,20191100000101,...,0,9999,999,81,NaN,NaN,NaN,NaN,NaN,NaN
1,2021,1,1,1805.3669,20191100000100,NaN,NaN,2,2336.9526,20191100000102,...,4150,4150,999,73,Service Occupations,"Hosts and hostesses, restaurant, lounge, and c...",35-9031,"Hosts and Hostesses, Restaurant, Lounge, and C...",432690.0,31200.0
2,2021,1,1,1805.3669,20191100000100,NaN,NaN,3,2129.3099,20191100000103,...,4110,4110,39,111,Service Occupations,Waiters and waitresses,35-3031,Waiters and Waitresses,2270910.0,35230.0
3,2021,2,1,1903.8219,20201100000100,NaN,NaN,1,1903.8219,20201100000101,...,0,9999,999,123,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,3,1,2738.9084,20191200000300,NaN,NaN,1,1570.4995,20191200000301,...,3602,4610,999,73,Service Occupations,Personal care aides,31-1122,NaN,4305810.0,35800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,69172,10,5158.6941,20250706859600,NaN,NaN,2,5501.4433,20250706859602,...,850,850,999,111,"Management, Business, Science, and Arts Occup...",Personal financial advisors,13-2052,Personal Financial Advisors,266800.0,105070.0
7724467,2026,69173,10,4778.5221,20250806883900,NaN,NaN,1,4778.5221,20250806883901,...,3930,3930,999,123,Service Occupations,Security guards and gambling surveillance offi...,33-9030,NaN,1292990.0,40695.0
7724468,2026,69173,10,4778.5221,20250806883900,NaN,NaN,2,5557.7384,20250806883902,...,3603,3600,999,92,Service Occupations,Nursing assistants,31-1131,Nursing Assistants,1448910.0,42260.0
7724469,2026,69174,10,4764.5418,20251107001400,NaN,NaN,1,4764.5418,20251107001401,...,2014,2010,999,123,"Management, Business, Science, and Arts Occup...","Social workers, all other",21-1029,"Social Workers, All Other",62930.0,71900.0


In [19]:
def merge_report(left, right, name, **kw):
    n0 = len(left)
    out = left.merge(right, how="left", validate="many_to_one", indicator="_m", **kw)
    sin = (out["_m"] == "left_only").sum()
    out = out.drop(columns="_m")
    print(f"{name}: antes={n0:,} | después={len(out):,} | filas nuevas por duplicados={len(out)-n0:,} "
          f"| sin pareja={sin:,} ({100*sin/len(out):.1f}%)")
    return out

# 1) SOC de 7 caracteres en la tabla de IA; varios O*NET (.00, .01...) -> un SOC
ia_cols = [c for c in df_ia.columns if c.startswith(("dv_rating", "human_rating"))]
df_ia["SOC_Code"] = df_ia["O*NET-SOC Code"].astype(str).str.strip().str[:7]
ia_soc = df_ia.groupby("SOC_Code", as_index=False)[ia_cols].mean()

# 2) Cruce exacto
df_completo = merge_report(df_final, ia_soc, "CPS x IA (exacto)", on="SOC_Code")
sin = df_completo[ia_cols[0]].isna()
print(df_completo.loc[sin, "SOC_Code"].value_counts().head(10))   # qué SOC quedan sin pareja

# 3) Respaldo por prefijo SOLO para las filas sin pareja
ia_6 = df_ia.assign(p=df_ia["SOC_Code"].str[:6]).groupby("p")[ia_cols].mean()
p = df_completo["SOC_Code"].str[:6]
for c in ia_cols:
    df_completo[c] = df_completo[c].fillna(p.map(ia_6[c]))
print(f"rellenadas por prefijo: {sin.sum() - df_completo[ia_cols[0]].isna().sum():,} "
      f"| sin dato final: {df_completo[ia_cols[0]].isna().sum():,}")

CPS x IA (exacto): antes=7,724,471 | después=7,724,471 | filas nuevas por duplicados=0 | sin pareja=5,004,461 (64.8%)
SOC_Code
53-3030    85529
25-2020    82749
41-2010    64836
37-201X    55677
35-2010    49160
25-9040    34712
51-91XX    29459
25-1000    27118
41-4010    26464
41-9020    25362
Name: count, dtype: int64
rellenadas por prefijo: 894,279 | sin dato final: 4,110,182


In [20]:
df_aplicables = df_completo.query("OCC!=0 & EMPSTAT<30 & EMPSTAT!=1").copy()
# Cambiamos los nombres para no perdernos
nombres = {
    "YEAR": "anio", "MONTH": "mes", "SERIAL": "id_hogar",
    "PERNUM": "num_persona_hogar", "CPSID": "id_hogar_longitudinal",
    "CPSIDP": "id_persona_longitudinal", "CPSIDV": "id_persona_validado",
    "WTFINL": "peso_persona", "HWTFINL": "peso_hogar",
    "ASECFLAG": "es_suplemento_asec", "ASECWT": "peso_persona_asec",
    "ASECWTH": "peso_hogar_asec",
    "AGE": "edad", "SEX": "sexo", "EMPSTAT": "estado_laboral",
    "DURUNEMP": "semanas_desempleado", "EDUC": "nivel_educativo",
    "OCC": "cod_ocupacion_2018", "OCC2010": "cod_ocupacion_2010",
    "Major_Group": "grupo_ocupacional", "Census_Title": "ocupacion_censo",
    "SOC_Code": "cod_soc", "OCC_TITLE": "ocupacion_bls",
    "TOT_EMP": "empleo_total_ocupacion", "A_MEDIAN": "salario_mediano_anual",
    "Title": "ocupacion_estudio_ia",
    "dv_rating_alpha": "exposicion_gpt4_estricta",
    "dv_rating_beta": "exposicion_gpt4_intermedia",
    "dv_rating_gamma": "exposicion_gpt4_maxima",
    "human_rating_alpha": "exposicion_humanos_estricta",
    "human_rating_beta": "exposicion_humanos_intermedia",
    "human_rating_gamma": "exposicion_humanos_maxima",
}

df_aplicables_base = df_aplicables.rename(columns=nombres)
df_aplicables_base.columns

Index(['anio', 'id_hogar', 'mes', 'peso_hogar', 'id_hogar_longitudinal',
       'es_suplemento_asec', 'peso_hogar_asec', 'num_persona_hogar',
       'peso_persona', 'id_persona_longitudinal', 'id_persona_validado',
       'peso_persona_asec', 'edad', 'sexo', 'estado_laboral',
       'cod_ocupacion_2018', 'cod_ocupacion_2010', 'semanas_desempleado',
       'nivel_educativo', 'grupo_ocupacional', 'ocupacion_censo', 'cod_soc',
       'ocupacion_bls', 'empleo_total_ocupacion', 'salario_mediano_anual',
       'exposicion_gpt4_estricta', 'exposicion_gpt4_intermedia',
       'exposicion_gpt4_maxima', 'exposicion_humanos_estricta',
       'exposicion_humanos_intermedia', 'exposicion_humanos_maxima'],
      dtype='str')

In [21]:
# Descartamos las columnas que no usaremos
df_aplicables_base = df_aplicables_base.drop(columns=["id_hogar", "num_persona_hogar", "id_hogar_longitudinal",
    "id_persona_validado", "peso_hogar", "es_suplemento_asec", "peso_persona_asec", "peso_hogar_asec"])
df_aplicables_base

,anio,mes,peso_persona,id_persona_longitudinal,edad,sexo,estado_laboral,cod_ocupacion_2018,cod_ocupacion_2010,semanas_desempleado,...,cod_soc,ocupacion_bls,empleo_total_ocupacion,salario_mediano_anual,exposicion_gpt4_estricta,exposicion_gpt4_intermedia,exposicion_gpt4_maxima,exposicion_humanos_estricta,exposicion_humanos_intermedia,exposicion_humanos_maxima
1,2021,1,2336.9526,20191100000102,22,1,10,4150,4150,999,...,35-9031,"Hosts and Hostesses, Restaurant, Lounge, and C...",432690.0,31200.0,0.057143,0.228571,0.400000,0.200000,0.328571,0.457143
2,2021,1,2129.3099,20191100000103,49,2,21,4110,4110,39,...,35-3031,Waiters and Waitresses,2270910.0,35230.0,0.081633,0.183673,0.285714,0.204082,0.224490,0.244898
4,2021,1,1570.4995,20191200000301,42,1,10,3602,4610,999,...,31-1122,NaN,4305810.0,35800.0,0.166667,0.222222,0.277778,0.222222,0.333333,0.444444
5,2021,1,2738.9084,20191200000302,39,2,10,5400,5400,999,...,43-4171,Receptionists and Information Clerks,910180.0,38010.0,0.200000,0.533333,0.866667,0.433333,0.583333,0.733333
8,2021,1,2036.5968,20210100000402,40,1,10,3321,3320,999,...,29-2031,Cardiovascular Technologists and Technicians,62960.0,74310.0,0.085714,0.357143,0.628571,0.114286,0.171429,0.228571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7724466,2026,10,5501.4433,20250706859602,55,1,10,850,850,999,...,13-2052,Personal Financial Advisors,266800.0,105070.0,0.052632,0.500000,0.947368,0.394737,0.671053,0.947368
7724467,2026,10,4778.5221,20250806883901,43,1,10,3930,3930,999,...,33-9030,NaN,1292990.0,40695.0,0.208333,0.302083,0.395833,0.083333,0.281250,0.479167
7724468,2026,10,5557.7384,20250806883902,39,2,10,3603,3600,999,...,31-1131,Nursing Assistants,1448910.0,42260.0,0.050847,0.101695,0.152542,0.033898,0.135593,0.237288
7724469,2026,10,4764.5418,20251107001401,48,1,10,2014,2010,999,...,21-1029,"Social Workers, All Other",62930.0,71900.0,0.029464,0.285813,0.542163,0.175198,0.358681,0.542163


In [22]:
df_aplicables_base.columns

Index(['anio', 'mes', 'peso_persona', 'id_persona_longitudinal', 'edad',
       'sexo', 'estado_laboral', 'cod_ocupacion_2018', 'cod_ocupacion_2010',
       'semanas_desempleado', 'nivel_educativo', 'grupo_ocupacional',
       'ocupacion_censo', 'cod_soc', 'ocupacion_bls', 'empleo_total_ocupacion',
       'salario_mediano_anual', 'exposicion_gpt4_estricta',
       'exposicion_gpt4_intermedia', 'exposicion_gpt4_maxima',
       'exposicion_humanos_estricta', 'exposicion_humanos_intermedia',
       'exposicion_humanos_maxima'],
      dtype='str')

In [23]:
df_aplicables_base["nivel_exposicion"]=df_aplicables_base["cod_soc"].map(
    pd.qcut(
        df_aplicables_base
        .drop_duplicates("cod_soc")
        .set_index("cod_soc")["exposicion_humanos_intermedia"],
        3,
        labels=["Baja", "Media", "Alta"]
    )
)

In [24]:
df_aplicables_base["desempleado"]=df_aplicables_base["estado_laboral"].isin([20, 21, 22]).astype(int)
df_aplicables_base["peso_desemp"]=df_aplicables_base["peso_persona"]*df_aplicables_base["desempleado"]

#Creamos tabla para primera grafica: ver como a evolucionado el desempleo al pasar de los años (2021-2022) según la expocisión de la ocupación a la IA (Alta, Media, Baja)
desempleo_nivel=(df_aplicables_base.groupby(["anio", "nivel_exposicion"], as_index=False, observed=True)
                 .agg(desempleados=("peso_desemp", "sum"),
                      fuerza_laboral=("peso_persona", "sum"))
                 )
desempleo_nivel["tasa"]=100*desempleo_nivel["desempleados"]/desempleo_nivel["fuerza_laboral"]
desempleo_nivel.pivot(index="anio", columns="nivel_exposicion", values="tasa").round(2)

nivel_exposicion,Baja,Media,Alta
anio,,,
2021,7.62,4.98,3.37
2022,5.09,3.38,2.24
2023,5.03,3.29,2.33
2024,5.32,3.50,2.77
2025,5.62,3.65,2.93
2026,5.65,3.52,2.95


In [25]:
indice = desempleo_nivel.pivot(index="anio", columns="nivel_exposicion", values="tasa")
indice = (100 * indice / indice.loc[2022]).round(1)
indice

nivel_exposicion,Baja,Media,Alta
anio,,,
2021,149.8,147.3,150.3
2022,100.0,100.0,100.0
2023,98.9,97.3,103.7
2024,104.6,103.4,123.4
2025,110.5,107.9,130.4
2026,111.1,104.1,131.6


In [26]:
px.line(
    (indice - 100).reset_index().melt(id_vars="anio", var_name="nivel_exposicion", value_name="indice"),
    x="anio", y="indice", color="nivel_exposicion", markers=True,
    labels={"indice": "Cambio en el desempleo respecto a 2022 (%)", "anio": "Año", "nivel_exposicion": "Exposición a la IA"},
).update_yaxes(ticksuffix="%")

En base a esta primera tabla, se decidio que se tomaría de referencia el año 2022, pues fue cuando se salio de la pandemia, además se empezó a recuperar la economía. Usamos la formula del indice, usando como referencia la tasa de desempleo de 2022, esto nos permite concluir lo siguiente: Si bien en aquellos empleos con mayor exposición a ser reemplazados por IA subio el desempleo en un 31% desde el 2022, las ocupaciones con menor expocision tambien tuvieron mayor desempleo que los de exposición media, 11.1% y 4.1% repectivamente. Esto nos demuestra que puede que el desempleo en los últimos años no este completamente causado por la evolución de la IA, pero que si se ve una relación entre el desempleo en las ocupaciones que más se pueden automatizar con IA y el desempleo desde la pandemia.


In [27]:
#Creamos 2da tabla que mide cuales son las ocupaciones con mas desempleo.
desempleo_ocupacion=(df_aplicables_base[df_aplicables_base["anio"]>=2022].groupby(["ocupacion_censo", "nivel_exposicion"], as_index=False, observed=True)
                     .agg(desempleados=("peso_desemp", "sum"),
                          fuerza_laboral=("peso_persona", "sum"),
                          n_desempleados=("desempleado", "sum"))
)

desempleo_ocupacion["tasa"]=100*desempleo_ocupacion["desempleados"]/desempleo_ocupacion["fuerza_laboral"]
desempleo_ocupacion.sort_values("tasa", ascending=False).head(15).round(2)
top = desempleo_ocupacion[desempleo_ocupacion["n_desempleados"] >= 100].nlargest(15, "tasa")
top

,ocupacion_censo,nivel_exposicion,desempleados,fuerza_laboral,n_desempleados,tasa
1,Actors,Media,8.302912e+05,3.731521e+06,241,22.250746
210,"Graders and sorters, agricultural products",Baja,3.911409e+05,3.200361e+06,124,12.221773
112,Crossing guards and flaggers,Media,3.071306e+05,3.212521e+06,104,9.560424
337,"Packers and packagers, hand",Baja,4.030189e+06,4.246022e+07,1079,9.491682
132,Dishwashers,Baja,1.554807e+06,1.661239e+07,516,9.359322
400,Recreation workers,Media,9.622436e+05,1.059353e+07,265,9.083310
473,Transportation service attendants,Baja,4.556490e+05,5.204842e+06,136,8.754328
129,Dining room and cafeteria attendants and barte...,Baja,1.548639e+06,1.808167e+07,444,8.564691
454,Tax preparers,Alta,6.371237e+05,7.523027e+06,162,8.468981
136,"Drywall installers, ceiling tile installers, a...",Baja,7.152347e+05,8.689693e+06,231,8.230839


In [28]:
px.bar(
    top.sort_values("tasa"),
    x="tasa", y="ocupacion_censo", color="nivel_exposicion", orientation="h",
    labels={"tasa": "Tasa de desempleo (%)", "ocupacion_censo": "", "nivel_exposicion": "Exposición a la IA"},
    height=550,
).update_xaxes(ticksuffix="%")

En esta segunda tabla y grafica, evaluamos las 15 ocupaciones con mayor desempleo. Cuando filtramos, descartamos las que contaba con menos de 100 desempleados, esocigmos este numero como referencia porque cuando las ocupaciones tienen menos de 100 desempleados, la tasa de desempleo suele ser bastante insetable, pues 1 mas o 1 menos puede cambiar el porcentage de la tasa considerablemente. A parttir del resultados, notamos que las labores con más desempleo son aquellas con expocisión a la IA baja y media, pues solo hay 1 de expocision alta. Si nos figamos en cuales son, podemos notar que parece que son aquellas que varían por estaciones, epocas y demandas, esto sugiere que el nivel de desempleo de una ocupación depende sobre todo de la naturaleza de ese trabajo, y no de su exposición a la IA. Sin embargo, este gráfico compara niveles y no permite evaluar el efecto de la IA. Para eso es necesario ver cómo cambió el desempleo a lo largo del tiempo, que es lo que se analiza en los siguientes gráficos.

In [29]:
#Para la tercera tabla, queremos ver cuales son las ocupaciones con expocision mas alta y cuales las que tienen expocision mas baja. No se tiene en cuenta el desempleo.
expo_ocupacion=(df_aplicables_base.groupby("ocupacion_censo", as_index=False, observed=True)
                .agg(exposicion_humanos_intermedia=("exposicion_humanos_intermedia", "first"),
                     empleo_total_ocupacion=("empleo_total_ocupacion", "first"),
                     nivel_exposicion=("nivel_exposicion", "first")))
expo_ocupacion.nlargest(10, "exposicion_humanos_intermedia")
expo_ocupacion.nsmallest(10, "exposicion_humanos_intermedia")

,ocupacion_censo,exposicion_humanos_intermedia,empleo_total_ocupacion,nivel_exposicion
4,Adhesive bonding machine operators and tenders,0.0,11500.0,Baja
26,Athletes and sports competitors,0.0,15070.0,Baja
31,Automotive glass installers and repairers,0.0,20310.0,Baja
44,Boilermakers,0.0,10190.0,Baja
46,"Brickmasons, blockmasons, and stonemasons",0.0,60370.0,Baja
50,Bus and truck mechanics and diesel engine spec...,0.0,289960.0,Baja
74,Cleaners of vehicles and equipment,0.0,380430.0,Baja
131,Dining room and cafeteria attendants and barte...,0.0,542750.0,Baja
134,Dishwashers,0.0,477450.0,Baja
148,Electrical power-line installers and repairers,0.0,131070.0,Baja


In [30]:
print(expo_ocupacion["exposicion_humanos_intermedia"].nunique())
print(df_aplicables_base["ocupacion_censo"].nunique())

411
525


In [31]:
px.bar(
    pd.concat([
        expo_ocupacion.nlargest(10, "exposicion_humanos_intermedia"),
        expo_ocupacion.nsmallest(10, "exposicion_humanos_intermedia"),
    ]).sort_values("exposicion_humanos_intermedia"),
    x="exposicion_humanos_intermedia", y="ocupacion_censo",
    color="nivel_exposicion", orientation="h",
    hover_data=["empleo_total_ocupacion"],
    labels={"exposicion_humanos_intermedia": "Exposición a la IA (0 a 1)",
            "ocupacion_censo": "", "nivel_exposicion": "Nivel"},
    height=600,
)

A partir de esta tabla y gráfica, podemos notar cuales son las 10 ocupaciones con mas expocision a la IA y cuales las 10 con menor. En la gráfica se ve que las 10 menores ni siquiera tienen una barra minima. Con esto podemos observar que aquellas ocupaciones con mas expocision a la IA (valores entre 0.68 y 0.84) suelen ser las que se basan en redactar, traducir, expresar ideas a un publico o escribir nuevos textos. Por otro lado, las 10 menores(todas con valor 0.0), suelen ser aquellas que se basan en trabajos que solo se pueden hacer fisicos o presenciales, como manejar buses o vehiculos de construcciín, ser alguien encargado de mantenimiendo, mecanicos y demás. Sin embargo, una expocision alta no significa que la profesión vaya a desaparecer, pero si que varias de sus funciones se pueden automatizar.
Tambien, notamos que en esta agrupación hay 411 valores diferentes de expocisión y solo hay 525 ocupaciones en total. Esto nos muestra que hay varias ocupaciones con expocision a la IA iguales. Asi que al menos 114 ocupaciones tienen el mismo valor de expocision que otra. Por esto es que los 10 menores que se ven en la gráfica se escogieron de manera arbitraria, pues todos tienen el mismo valor, y tal vez se excluyeron esos en especifico porque estaban en ese orden en la tabla base. 

In [32]:
# Para la tabla y el gráfico 4 y 5, vamos a evaluar si el desempleo subio mas en las ocupaciones muy expuestas a la IA que en las demas según periodos de tiempo
df_aplicables_base["periodo"]=df_aplicables_base["anio"].map({2022:"antes", 2023: "antes", 2025:"despues", 2026:"despues"})
tasa_ocupacion_periodo=(df_aplicables_base.dropna(subset=["periodo"]).groupby(["ocupacion_censo", "nivel_exposicion", "periodo"], observed=True)
                        .agg(desempleados=("peso_desemp", "sum"),
                             fuerza_laboral=("peso_persona", "sum"),
                             n_desempleados=("desempleado", "sum")))

tasa_ocupacion_periodo["tasa"]=100*tasa_ocupacion_periodo["desempleados"]/tasa_ocupacion_periodo["fuerza_laboral"]

cambio_ocupacion=tasa_ocupacion_periodo[["tasa", "n_desempleados"]].unstack("periodo")
cambio_ocupacion.columns




MultiIndex([(          'tasa',   'antes'),
            (          'tasa', 'despues'),
            ('n_desempleados',   'antes'),
            ('n_desempleados', 'despues')],
           names=[None, 'periodo'])

In [33]:
cambio_ocupacion.columns = ["tasa_antes", "tasa_despues", "n_desempleados_antes", "n_desempleados_despues"]
cambio_ocupacion = cambio_ocupacion.dropna().reset_index()

cambio_ocupacion["n_total"] = cambio_ocupacion["n_desempleados_antes"] + cambio_ocupacion["n_desempleados_despues"]
cambio_ocupacion["cambio_pp"] = cambio_ocupacion["tasa_despues"] - cambio_ocupacion["tasa_antes"]
cambio_ocupacion = cambio_ocupacion[cambio_ocupacion["n_total"] >= 30]

cambio_ocupacion["trayectoria"] = np.select(
    [cambio_ocupacion["cambio_pp"] >= 1, cambio_ocupacion["cambio_pp"] <= -1],
    ["Subió", "Bajó"], default="Estable",
)
cambio_ocupacion = cambio_ocupacion.merge(
    expo_ocupacion[["ocupacion_censo", "exposicion_humanos_intermedia", "empleo_total_ocupacion"]],
    on="ocupacion_censo",
)
cambio_ocupacion.head()

,ocupacion_censo,nivel_exposicion,tasa_antes,tasa_despues,n_desempleados_antes,n_desempleados_despues,n_total,cambio_pp,trayectoria,exposicion_humanos_intermedia,empleo_total_ocupacion
0,Accountants and auditors,Alta,1.358908,1.929816,158,197,355,0.570908,Estable,0.520000,1449500.0
1,Actors,Media,21.407955,24.645477,111,81,192,3.237522,Subió,0.156250,55000.0
2,Advertising sales agents,Alta,2.781321,4.567234,22,29,51,1.785912,Subió,0.692308,91700.0
3,"Agents and business managers of artists, perfo...",Media,4.155698,6.494843,8,22,30,2.339145,Subió,0.369565,12620.0
4,Aircraft mechanics and service technicians,Baja,1.181129,3.655483,20,26,46,2.474354,Subió,0.059701,138090.0


In [34]:
cambio_ocupacion["nivel_exposicion"].value_counts()
pd.crosstab(cambio_ocupacion["nivel_exposicion"], cambio_ocupacion["trayectoria"], normalize="index").round(2)

trayectoria,Bajó,Estable,Subió
nivel_exposicion,,,
Baja,0.16,0.52,0.32
Media,0.11,0.57,0.33
Alta,0.08,0.50,0.42


In [35]:
cambio_ocupacion["exposicion_humanos_intermedia"].corr(cambio_ocupacion["cambio_pp"], method="spearman")

np.float64(0.11792940255095015)

In [36]:
px.scatter(
    cambio_ocupacion[cambio_ocupacion["nivel_exposicion"] == "Alta"],
    x="exposicion_humanos_intermedia", y="cambio_pp",
    color="trayectoria", size="empleo_total_ocupacion", hover_name="ocupacion_censo",
    labels={"exposicion_humanos_intermedia": "Exposición a la IA (0 a 1)",
            "cambio_pp": "Cambio en la tasa de desempleo (puntos)", "trayectoria": "Trayectoria"},
).add_hline(y=0)

In [37]:
px.box(cambio_ocupacion, x="nivel_exposicion", y="cambio_pp", points="all",
       hover_name="ocupacion_censo", category_orders={"nivel_exposicion": ["Baja", "Media", "Alta"]},
       labels={"cambio_pp": "Cambio en la tasa de desempleo (puntos)", "nivel_exposicion": "Exposición a la IA"})

Al evaluar solo las ocupaciones de exposición alta, la distribución se inclina hacia las que subieron su desempleo: alrededor del 42 % aumentó su tasa entre 2022-2023 y 2025-2026, mientras que solo el 8 % la redujo. La ocupación más expuesta, Interpreters and Translators, subió su tasa de desempleo en 1.13 puntos. Sin embargo, al comparar todas las ocupaciones, la correlación entre exposición y cambio en el desempleo es débil (0.11), por lo que una mayor exposición no se traduce claramente en un mayor aumento.

En la segunda gráfica se comparan los tres niveles. El de exposición baja es el más disperso y tiene los valores extremos, tanto la ocupación que más subió (Sailors and marine oilers, 13.4 puntos) como la que más bajó (Other ground maintenance workers, -10.5 puntos). Esta dispersión se debe probablemente a que varias de esas ocupaciones son pequeñas y su tasa es inestable. En el nivel medio, la mayoría de las ocupaciones se concentra cerca de 0, es decir, su desempleo se mantuvo estable. En el nivel alto, la mediana está algo por encima de 0 (0.7), se ve que mas de la mayoria de ocupaciones incrementaron en tasa de desempleo, solo unas cuantas bajaron. 

En conjunto, el desempleo aumentó algo más en las ocupaciones más expuestas, pero la diferencia es pequeña y no permite afirmar que la IA sea la causa.

In [38]:
cols = [
    "YEAR", "MONTH", "WTFINL", "AGE", "SEX", "EMPSTAT", "OCC", "DURUNEMP", "EDUC",
    "Major_Group", "Census_Title", "SOC_Code", "TOT_EMP", "A_MEDIAN",
    "human_rating_alpha", "human_rating_beta", "human_rating_gamma",
    "dv_rating_alpha", "dv_rating_beta", "dv_rating_gamma",
]

(df_completo.query("OCC != 0 & EMPSTAT < 30")[cols]
    .to_pickle("../data/processed/df_completo.pkl"))